In [30]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationSummaryBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough 
from langchain.callbacks import StreamingStdOutCallbackHandler


llm = ChatOpenAI(temperature=0.1,
                 streaming=True,
                 callbacks=[StreamingStdOutCallbackHandler()])

cache_dir = LocalFileStore("../../.cache")

memory = ConversationSummaryBufferMemory(
    llm=llm,
    return_messages=True,
    memory_key="chat_history",
    # output_key="answer"
)

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100
)

loader = UnstructuredFileLoader("../../files/chapter_01.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    memory=memory,
    chain_type="stuff",
    verbose=True
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "다음 내용을 토대로 답변하세요, {context}"),
        ("human", "{question}")
    ]
)

retriever = vectorstore.as_retriever()

chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

chain.invoke("Aaronson은 유죄인가요?")
chain.invoke("그가 테이블에 어떤 메시지를 썼나요?")
chain.invoke("Julia 는 누구인가요?")




# prompt = ChatPromptTemplate.from_messages(
#     [
#         ("system", "아래 내용을 참고해서 꾸미지 말고, 질문에 대한 답을 제시하세요."),
#         MessagesPlaceholder(variable_name="chat_history"),
#         # ("ai", "{context}"),
#         ("human", "{question}")
#     ]
# )


# # chain = (
# #     {
# #         "context": retriever,
# #         "question": RunnablePassthrough()
# #     }
# # ) | prompt | llm

# def load_memory(_):
#     return memory.load_memory_variables({})["chat_history"]

# chain = RunnablePassthrough.assign(chat_history=load_memory) | prompt | llm

# def invoke_chain(input):
#     result = chain.invoke({"question": input})
#     memory.save_context(
#         {"question": input},
#         {"context": result.content}
#     )

# invoke_chain("Aaronson은 유죄인가요?")
# invoke_chain("그가 테이블에 어떤 메시지를 썼나요?")
# invoke_chain("Julia 는 누구인가요?")





Created a chunk of size 963, which is longer than the specified 600
Created a chunk of size 774, which is longer than the specified 600
Created a chunk of size 954, which is longer than the specified 600
Created a chunk of size 922, which is longer than the specified 600
Created a chunk of size 1168, which is longer than the specified 600
Created a chunk of size 821, which is longer than the specified 600
Created a chunk of size 700, which is longer than the specified 600
Created a chunk of size 745, which is longer than the specified 600
Created a chunk of size 735, which is longer than the specified 600
Created a chunk of size 1110, which is longer than the specified 600
Created a chunk of size 991, which is longer than the specified 600
Created a chunk of size 990, which is longer than the specified 600
Created a chunk of size 1182, which is longer than the specified 600
Created a chunk of size 1491, which is longer than the specified 600
Created a chunk of size 1401, which is longe

아론슨(Aaronson)은 조지 오웰의 소설 '1984'에서 등장하는 인물로, 이야기 속에서는 아론슨과 그의 동료들이 반정부 활동으로 비난을 받고 처형되었다는 내용이 나옵니다. 이에 따라 아론슨은 유죄로 여겨지며 처형되었다고 볼 수 있습니다.그가 테이블에 쓴 메시지는 다음과 같습니다: "Suddenly he began writing in sheer panic, only imperfectly aware of what he was setting down. His small but childish handwriting straggled up and down the page, shedding first its capital letters and finally even its full stops."Julia는 위 문단에서 언급된 여성으로, 소설 1984년의 등장인물 중 하나입니다. 이 문단에서는 Winston이 처음으로 Julia를 만나는 장면이 묘사되어 있습니다. Julia는 Fiction Department에서 일하는 여성으로, Junior Anti-Sex League의 상징인 좁은 붉은색 띠를 허리에 감고 있습니다. Winston은 처음 봤을 때부터 Julia를 싫어했으며, 이는 그녀가 담배를 피우고 스패너를 들고 다니는 모습을 보았기 때문입니다.

AIMessageChunk(content='Julia는 위 문단에서 언급된 여성으로, 소설 1984년의 등장인물 중 하나입니다. 이 문단에서는 Winston이 처음으로 Julia를 만나는 장면이 묘사되어 있습니다. Julia는 Fiction Department에서 일하는 여성으로, Junior Anti-Sex League의 상징인 좁은 붉은색 띠를 허리에 감고 있습니다. Winston은 처음 봤을 때부터 Julia를 싫어했으며, 이는 그녀가 담배를 피우고 스패너를 들고 다니는 모습을 보았기 때문입니다.')